### Phase 1: Pre-Computation (Requires GPU to speed up the pre-computation pipeline)
1. Vectorization: Pre-compute dense embeddings for all 100,000 candidates using a fast, compact model

2. Behavioral Normalization: Extract and normalize the 23 behavioral signals (like recruiter_response_rate, last_active_date, and github_activity_score) into continuous scalar multipliers.

3. Artifact Creation: Bundle these pre-computed embeddings and normalized features into a compact local index.

#### Install libraries

In [1]:
!pip install sentence-transformers faiss-cpu pandas numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 98.1 MB/s eta 0:00:00


#### Import core data processing and vectorization libraries

In [2]:
import json
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
import faiss
import os
from datetime import datetime

##### CONFIGURATION & CONSTANTS

In [3]:
DATA_PATH = '/content/drive/MyDrive/Redrob_Startup/candidates.jsonl'
OUTPUT_DIR = '/content/drive/MyDrive/Redrob_Startup/'

TEST_LIMIT = None

# Baseline date to calculate inactivity based on the dataset generation timeframe
CURRENT_DATE = datetime(2026, 6, 7)

#### Behavorial Logic

In [4]:
def calculate_behavioral_multiplier(signals):
    # Converts raw behavioral signals into a single scalar multiplier to weigh the final score.
    multiplier = 1.0

    # Penalize ghosting recruiters; boost highly responsive candidates
    response_rate = signals.get('recruiter_response_rate', 0.5)
    if response_rate < 0.2: multiplier *= 0.7
    elif response_rate > 0.8: multiplier *= 1.1

    # Heavily penalize candidates who haven't logged in recently
    last_active = signals.get('last_active_date')
    if last_active:
        try:
            active_date = datetime.strptime(last_active, "%Y-%m-%d")
            days_inactive = (CURRENT_DATE - active_date).days
            if days_inactive > 180: multiplier *= 0.6
            elif days_inactive > 90: multiplier *= 0.85
            elif days_inactive < 14: multiplier *= 1.15
        except ValueError:
            pass

    # Penalize candidates who haven't filled out their profile
    if signals.get('profile_completeness_score', 50) < 40:
        multiplier *= 0.8

    # Boost "shippers" with high GitHub activity
    if signals.get('github_activity_score', -1) > 75:
        multiplier *= 1.1

    # Reward candidates who can join immediately; penalize long notice periods
    notice_days = signals.get('notice_period_days', 90)
    if notice_days <= 30: multiplier *= 1.1
    elif notice_days > 90: multiplier *= 0.9

    return multiplier

#### Data Extraction Loop

In [5]:
parsed_data = []
with open(DATA_PATH, 'r', encoding='utf-8') as file:
    for i, line in enumerate(file):
        if TEST_LIMIT and i >= TEST_LIMIT: break

        cand = json.loads(line.strip())
        c_id = cand.get('candidate_id')

        # Semantic Extraction: Pulling the actual text context, avoiding empty fields
        profile = cand.get('profile', {})
        headline = profile.get('headline', '')
        summary = profile.get('summary', '')

        # Safely extract skill names from the list of dictionaries
        skills_list = [s.get('name', '') for s in cand.get('skills', []) if isinstance(s, dict) and s.get('name')]
        skills_str = ", ".join(skills_list)

        # Extract actual role descriptions to bypass keyword-stuffing traps
        exp_parts = []
        for role in cand.get('career_history', []):
            title = role.get('title', '')
            company = role.get('company', '')
            desc = role.get('description', '')
            if title or desc:
                exp_parts.append(f"{title} at {company}: {desc}")
        exp_str = " | ".join(exp_parts)

        # Construct the dense document for the Bi-Encoder
        doc_parts = []
        if headline: doc_parts.append(f"Headline: {headline}")
        if summary: doc_parts.append(f"Summary: {summary}")
        if skills_str: doc_parts.append(f"Skills: {skills_str}")
        if exp_str: doc_parts.append(f"Experience: {exp_str}")
        semantic_text = " ".join(doc_parts)

        # Behavioral Extraction: Compute the multiplier offline to save CPU later
        signals = cand.get('redrob_signals', {})
        final_multiplier = calculate_behavioral_multiplier(signals)

        # Detect obvious honeypots like accepted offers but never completed an interview
        is_honeypot = 1 if signals.get('offer_acceptance_rate') == 1.0 and signals.get('interview_completion_rate') == 0.0 else 0

        # Store the clean, processed data
        parsed_data.append({
            'candidate_id': c_id,
            'semantic_text': semantic_text,
            'behavioral_multiplier': final_multiplier,
            'is_honeypot_flag': is_honeypot
        })

# Convert the parsed data into a pandas DataFrame
df = pd.DataFrame(parsed_data)
print(f"Parsed {len(df)} candidates. Generating embeddings...")

Parsed 100000 candidates. Generating embeddings...


#### Vectorization

In [6]:
model = SentenceTransformer('all-MiniLM-L6-v2')

# Generate 384-dimensional embeddings utilizing the Colab GPU
embeddings = model.encode(df['semantic_text'].tolist(), show_progress_bar=True)
embeddings = np.array(embeddings).astype('float32')

# --- 5. INDEXING & ARTIFACT CREATION ---
# Build an Inner Product FAISS index which is equivalent to Cosine Similarity when normalized
dimension = embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)
faiss.normalize_L2(embeddings)
index.add(embeddings)

# Drop the heavy text data to keep the metadata file strictly under the 5GB limit
metadata_df = df.drop(columns=['semantic_text'])

# Save the compiled FAISS index and metadata to Google Drive
faiss.write_index(index, os.path.join(OUTPUT_DIR, 'candidate_index.faiss'))
metadata_df.to_csv(os.path.join(OUTPUT_DIR, 'candidate_metadata.csv'), index=False)

print("\nPhase 1 Complete: FAISS index and metadata saved to Drive successfully.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/3125 [00:00<?, ?it/s]


Phase 1 Complete: FAISS index and metadata saved to Drive successfully.
